In [2]:
"""
================================================================================
  DATASET MAESTRO PARA MACHINE LEARNING — Red REMMAQ (Quito)
  Autor : Senior Data Engineer
  Modelo: HistGradientBoostingRegressor (scikit-learn)
================================================================================

  ESTRUCTURA DE LOS ARCHIVOS REMMAQ (idéntica al script EDA):
  ─────────────────────────────────────────────────────────────
  Cada archivo (PM10.xlsx, SO2.xlsx, CO.xlsx…) tiene esta forma:

        Fecha          │ BELISARIO │ CARAPUNGO │ COTOCOLLAO │ …
    ───────────────────┼───────────┼───────────┼────────────┼──
    2004-01-01 00:00   │    NaN    │   12.5    │    8.3     │
    2004-01-01 01:00   │   14.2   │   NaN     │    9.1     │
    …

  El script extrae UNA columna (la parroquia objetivo) de cada archivo
  y las une en un único DataFrame, exactamente igual que el EDA original.
  No existen archivos separados por parroquia.

  REGLAS DE FILTRADO ML (en orden de aplicación):
  ─────────────────────────────────────────────────
  R1 — Sin target: elimina filas donde PM2.5 es NaN.
  R2 — Coherencia química: elimina filas donde TODOS los contaminantes
       son NaN simultáneamente (falla total de sensores).
  R3 — Gaps > 24 h: elimina bloques continuos sin ningún contaminante.
  R4 — NaNs secundarios: se preservan para HistGradientBoosting nativo.

  SALIDA:
  ─────────────────────────────────────────────────
  dataset_ml_[parroquia].csv  — listo para entrenar sin imputación previa.
================================================================================
"""

import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# ① CONFIGURACIÓN — edita aquí antes de ejecutar
# ══════════════════════════════════════════════════════════════════════════════

# Parroquia/estación objetivo.  Cambia este valor para procesar otra estación.
# El script busca este nombre (sin distinción de mayúsculas) como columna
# dentro de cada archivo — no necesitas archivos separados por parroquia.
PARROQUIA_OBJETIVO: str = "BELISARIO"

# Variable a predecir (target). Si el archivo no existe, el script avisa.
TARGET: str = "PM25"

# Contaminantes disponibles. Se usan para las Reglas R2 y R3.
CONTAMINANTES: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]

# Excluir el período completo de pandemia COVID-19 (01-01-2020 → 31-12-2021)
EXCLUIR_PANDEMIA: bool = True
PANDEMIA_INICIO: str   = "2020-01-01"   # ← modificado: todo 2020 y 2021
PANDEMIA_FIN:    str   = "2021-12-31"

# Horas de silencio consecutivo para considerar "falla de estación" (R3)
UMBRAL_GAP_HORAS: int = 24

# Rangos físicos aceptables. Valores fuera de rango → NaN (errores de sensor)
RANGOS_VALIDOS: dict[str, tuple[float, float]] = {
    "PM25":            (0, 400),   # ← ajustado: máximo 400 µg/m³
    "PM10":            (0, 999),
    "O3":              (0, 500),
    "CO":              (0,  50),
    "NO2":             (0, 500),
    "SO2":             (0, 500),
    "Temperatura":     (-10, 50),
    "Humedad":         (0, 100),
    "Viento_Velocidad":(0,  50),
    "Viento_Direccion":(0, 360),
    "Precipitacion":   (0, 200),
}

# Nombres de archivo esperados para cada variable
# (el script los busca en orden y usa el primero que exista)
NOMBRES_ARCHIVOS: dict[str, list[str]] = {
    "PM25":             ["PM2.5.xlsx", "PM25.xlsx", "PM25.csv"],
    "PM10":             ["PM10.xlsx",  "PM10.csv"],
    "O3":               ["O3.xlsx",    "O3.csv"],
    "CO":               ["CO.xlsx",    "CO.csv"],
    "NO2":              ["NO2.xlsx",   "NO2.csv"],
    "SO2":              ["SO2.xlsx",   "SO2.csv"],
    "Temperatura":      ["TMP.xlsx",   "Temperatura.xlsx", "Temperatura.csv"],
    "Humedad":          ["HUM.xlsx",   "Humedad.xlsx",     "Humedad.csv"],
    "Viento_Velocidad": ["VEL.xlsx",   "Viento_Velocidad.xlsx"],
    "Viento_Direccion": ["DIR.xlsx",   "Viento_Direccion.xlsx"],
    "Precipitacion":    ["LLU.xlsx",   "Precipitacion.xlsx"],
}

# Carpeta de salida
OUTPUT_DIR: str = "dataset_ml"


# ══════════════════════════════════════════════════════════════════════════════
# ② CARGA DE ARCHIVOS
#    Idéntica al EDA original: un archivo por variable, N columnas por parroquia
# ══════════════════════════════════════════════════════════════════════════════

def _leer_archivo(path: Path) -> pd.DataFrame:
    """Lee CSV o Excel y devuelve el DataFrame crudo."""
    ext = path.suffix.lower()
    if ext == ".csv":
        for sep in (",", ";", "\t"):
            try:
                df = pd.read_csv(path, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
        raise ValueError(f"No se pudo leer el CSV: {path}")
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Extensión no soportada: {ext}")


def _extraer_columna_parroquia(path: Path,
                                variable: str,
                                parroquia: str) -> pd.Series:
    """
    Carga el archivo y extrae la columna de la parroquia solicitada.

    Pasos (idénticos al EDA original):
      1. Renombrar la primera columna como 'Fecha'.
      2. Eliminar la fila de unidades (si la hay).
      3. Convertir el índice a datetime.
      4. Buscar la columna de la parroquia (exacta primero, parcial después).
      5. Convertir a numérico y devolver como pd.Series con nombre = variable.
    """
    if not path.exists():
        print(f"  ✗  [{variable:<22}] No encontrado: {path.name}")
        return pd.Series(dtype=float, name=variable)

    df = _leer_archivo(path)

    # ── Columna de fecha ─────────────────────────────────────────────────────
    df = df.rename(columns={df.columns[0]: "Fecha"})

    # Eliminar fila de unidades (ug/m3, °C, etc.)
    mask_u = df["Fecha"].astype(str).str.contains(
        r"unidad|unit|ug|mg|%|m/s|°|grados", case=False, na=False, regex=True
    )
    df = df[~mask_u].reset_index(drop=True)

    # Convertir Fecha → datetime
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df = df.dropna(subset=["Fecha"]).set_index("Fecha").sort_index()
    df.columns = [str(c).strip() for c in df.columns]  # limpiar espacios

    # ── Buscar columna de la parroquia ───────────────────────────────────────
    pu        = parroquia.strip().upper()
    col_match = None

    # 1) Coincidencia exacta (case-insensitive)
    for col in df.columns:
        if col.upper() == pu:
            col_match = col
            break

    # 2) Coincidencia parcial
    if col_match is None:
        candidatos = [c for c in df.columns if pu in c.upper()]
        if candidatos:
            col_match = candidatos[0]
            print(f"  ⚠  [{variable:<22}] '{parroquia}' parcial → '{col_match}'")

    if col_match is None:
        print(f"  ✗  [{variable:<22}] '{parroquia}' no encontrada.")
        print(f"     Parroquias disponibles: {', '.join(df.columns.tolist())}")
        return pd.Series(dtype=float, name=variable)

    serie = pd.to_numeric(df[col_match], errors="coerce")
    serie.name = variable
    n_val = serie.notna().sum()
    rango = f"{serie.index.min().date()} → {serie.index.max().date()}"
    print(f"  ✓  [{variable:<22}] {n_val:>8,} valores  |  {rango}  |  "
          f"col='{col_match}'")
    return serie


def buscar_archivos_en_directorio(data_dir: Path) -> dict[str, Path]:
    """
    Busca automáticamente los archivos de cada variable en el directorio.
    Usa NOMBRES_ARCHIVOS para saber qué buscar.
    Devuelve solo los archivos que realmente existen.
    """
    encontrados: dict[str, Path] = {}
    for variable, candidatos in NOMBRES_ARCHIVOS.items():
        for nombre in candidatos:
            ruta = data_dir / nombre
            if ruta.exists():
                encontrados[variable] = ruta
                break
    return encontrados


def cargar_dataset_parroquia(archivos_encontrados: dict[str, Path],
                              parroquia: str) -> pd.DataFrame:
    """
    Carga un archivo por cada variable disponible, extrae la columna
    de la parroquia y une todo con OUTER JOIN sobre el Timestamp.

    ¿Por qué outer join?
    ────────────────────
    • Cada archivo puede tener distinto rango de fechas (ej: PM10 desde 2013,
      SO2 desde 2004).
    • El outer join crea UNA fila por cada timestamp que aparezca en cualquier
      archivo. Los valores ausentes quedan NaN, no se eliminan.
    • Esto garantiza que ningún dato válido se pierde en el merge.
      Solo los filtros posteriores eliminan filas, con trazabilidad completa.
    """
    series: list[pd.Series] = []

    for variable, ruta in archivos_encontrados.items():
        s = _extraer_columna_parroquia(ruta, variable, parroquia)
        if not s.empty:
            series.append(s)

    if not series:
        raise RuntimeError(
            f"No se encontró '{parroquia}' en ningún archivo. "
            "Revisa el nombre de la parroquia."
        )

    df = pd.concat(series, axis=1, join="outer").sort_index()
    df.index.name = "Timestamp"
    return df


# ══════════════════════════════════════════════════════════════════════════════
# ③ LIMPIEZA DE ERRORES DE SENSOR
# ══════════════════════════════════════════════════════════════════════════════

def limpiar_errores_sensor(df: pd.DataFrame,
                            rangos: dict[str, tuple[float, float]]
                            ) -> pd.DataFrame:
    """
    Reemplaza con NaN:
      • Códigos de error (-999, -9999, 9999)
      • Valores fuera del rango físico aceptable
    Debe ejecutarse ANTES de las reglas de filtrado ML.
    """
    df_c = df.copy()
    reporte = []

    for col in df_c.columns:
        n0 = df_c[col].notna().sum()

        # Códigos de error de sensor
        df_c.loc[df_c[col].isin([-999, -9999, 9999]), col] = np.nan

        # Rango físico
        rng = rangos.get(col)
        if rng:
            vmin, vmax = rng
            df_c.loc[(df_c[col] < vmin) | (df_c[col] > vmax), col] = np.nan

        n_elim = n0 - df_c[col].notna().sum()
        if n_elim > 0:
            reporte.append(f"    • {col:<22} {n_elim:>7,} valores anómalos → NaN")

    if reporte:
        print("\n  🔧  Errores de sensor corregidos:")
        for l in reporte:
            print(l)
    else:
        print("\n  ✅  Sin errores de sensor detectados.")

    return df_c


# ══════════════════════════════════════════════════════════════════════════════
# ④ FILTROS DE COHERENCIA ML
# ══════════════════════════════════════════════════════════════════════════════

class _Auditor:
    """Registra y muestra el conteo de filas eliminadas en cada etapa."""

    def __init__(self, n_total: int) -> None:
        self.n_total  = n_total
        self._etapas: list[tuple[str, int]] = []

    def registrar(self, etapa: str, n_antes: int, n_despues: int) -> None:
        eliminadas = n_antes - n_despues
        self._etapas.append((etapa, eliminadas))
        pct = eliminadas / self.n_total * 100 if self.n_total else 0
        icono = "✂" if eliminadas else "✓"
        print(f"  {icono}  {etapa:<48}  "
              f"−{eliminadas:>8,} filas  ({pct:5.1f}%)")

    def resumen(self) -> None:
        n_final    = self.n_total - sum(e for _, e in self._etapas)
        retenidas  = n_final / self.n_total * 100 if self.n_total else 0
        sep        = "═" * 70
        print(f"\n{sep}")
        print(f"  RESUMEN DE FILTRADO")
        print(f"{'─'*70}")
        print(f"  {'Filas originales (post-merge)':<48}  {self.n_total:>10,}")
        for etapa, elim in self._etapas:
            print(f"  {etapa:<48}  −{elim:>9,}")
        print(f"{'─'*70}")
        print(f"  {'Filas en dataset final':<48}  {n_final:>10,}  "
              f"({retenidas:.1f}% del original)")
        print(f"{sep}\n")


def _r1_sin_target(df: pd.DataFrame, target: str,
                   aud: _Auditor) -> pd.DataFrame:
    """
    R1 — Elimina filas donde el TARGET (PM25) es NaN.

    Sin el valor a predecir, la fila no puede usarse en entrenamiento
    supervisado. Es la regla más restrictiva y la primera en aplicarse.
    """
    if target not in df.columns:
        print(f"  ⚠  R1: la columna '{target}' no existe — omitida.")
        return df
    n = len(df)
    df_out = df.dropna(subset=[target]).copy()
    aud.registrar(f"R1 — Filas sin target ({target}=NaN)", n, len(df_out))
    return df_out


def _r2_sin_referencia_quimica(df: pd.DataFrame,
                                contaminantes: list[str],
                                aud: _Auditor) -> pd.DataFrame:
    """
    R2 — Elimina filas donde TODOS los contaminantes son NaN a la vez.

    Tras R1, el target siempre tiene dato. Esta regla detecta filas donde
    el resto de contaminantes (PM10, O3, CO, NO2, SO2) son todos NaN, lo
    que indica una falla simultánea de sensores. Sin referencia química del
    período, el modelo no puede aprender el contexto de contaminación.

    Solo se evalúan las columnas que realmente existan en el DataFrame.
    """
    cols = [c for c in contaminantes if c in df.columns]
    if len(cols) < 2:
        print("  ⚠  R2: menos de 2 contaminantes — omitida.")
        return df
    n = len(df)
    # Fila válida = al menos UN contaminante con dato
    valida = df[cols].notna().any(axis=1)
    df_out = df[valida].copy()
    aud.registrar("R2 — Fila sin ningún contaminante (falla total)", n, len(df_out))
    return df_out


def _r3_gaps_largos(df: pd.DataFrame,
                     contaminantes: list[str],
                     umbral_h: int,
                     aud: _Auditor) -> pd.DataFrame:
    """
    R3 — Elimina bloques > umbral_h horas donde ningún contaminante reporta.

    Algoritmo O(n):
      1. 'silencio' = True en cada fila donde TODOS los contaminantes son NaN.
      2. Agrupa filas consecutivas de silencio con cumsum sobre cambios de estado.
      3. Para cada bloque de silencio calcula duración real en horas.
      4. Marca los bloques cuya duración excede el umbral.
      5. Elimina esas filas del DataFrame.

    Usa duración real (horas entre timestamps) en vez de conteo de filas,
    para ser robusto ante índices irregulares o con dobles entradas.
    """
    cols = [c for c in contaminantes if c in df.columns]
    if not cols:
        print("  ⚠  R3: sin columnas de contaminantes — omitida.")
        return df

    n        = len(df)
    silencio = df[cols].isna().all(axis=1)          # True donde no hay ningún contaminante
    cambio   = silencio != silencio.shift()          # True en el inicio de cada nuevo bloque
    id_blq   = cambio.cumsum()                       # ID único por bloque consecutivo

    # Calcular duración de los bloques de silencio
    df_sil = df[silencio].copy()
    df_sil["_blq_"] = id_blq[silencio]

    if df_sil.empty:
        aud.registrar(f"R3 — Gaps >{umbral_h}h (ninguno detectado)", n, n)
        return df

    duraciones = df_sil.groupby("_blq_").apply(
        lambda g: (g.index.max() - g.index.min()).total_seconds() / 3600
    )
    blqs_malos = set(duraciones[duraciones > umbral_h].index.tolist())

    if not blqs_malos:
        aud.registrar(f"R3 — Gaps >{umbral_h}h (ninguno detectado)", n, n)
        return df

    # Mostrar detalle de gaps encontrados
    print(f"\n    Gaps de silencio detectados (>{umbral_h}h):")
    for bid in sorted(blqs_malos):
        g      = df_sil[df_sil["_blq_"] == bid]
        dur_h  = duraciones[bid]
        print(f"      • {g.index.min()}  →  {g.index.max()}  ({dur_h:.1f} h)")

    mascara    = silencio & id_blq.isin(blqs_malos)
    df_out     = df[~mascara].copy()
    aud.registrar(
        f"R3 — Gaps >{umbral_h}h ({len(blqs_malos)} bloque/s)",
        n, len(df_out)
    )
    return df_out


def _filtrar_pandemia(df: pd.DataFrame, inicio: str, fin: str,
                       aud: _Auditor) -> pd.DataFrame:
    """Excluye el período pandemia COVID-19 del dataset."""
    n    = len(df)
    mask = (df.index >= inicio) & (df.index <= fin)
    df_c = df[~mask].copy()
    aud.registrar(f"Pandemia excluida ({inicio} → {fin})", n, len(df_c))
    return df_c


# ══════════════════════════════════════════════════════════════════════════════
# ⑤ FEATURES ADICIONALES PARA MACHINE LEARNING
# ══════════════════════════════════════════════════════════════════════════════

def _generar_features_ml(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Añade al dataset:
      - Lags del target: PM25_lag_1h, PM25_lag_3h, PM25_lag_24h
      - Variables cíclicas temporales: hora_sin, hora_cos, mes_sin, mes_cos
    """
    df_feat = df.copy()

    # Lags del target (filas pasadas)
    df_feat["PM25_lag_1h"]  = df_feat[target].shift(1)
    df_feat["PM25_lag_3h"]  = df_feat[target].shift(3)
    df_feat["PM25_lag_24h"] = df_feat[target].shift(24)

    # Características cíclicas de hora (0-23)
    hour = df_feat.index.hour
    df_feat["hora_sin"] = np.sin(2 * np.pi * hour / 24)
    df_feat["hora_cos"] = np.cos(2 * np.pi * hour / 24)

    # Características cíclicas de mes (1-12)
    month = df_feat.index.month
    df_feat["mes_sin"] = np.sin(2 * np.pi * month / 12)
    df_feat["mes_cos"] = np.cos(2 * np.pi * month / 12)

    return df_feat


# ══════════════════════════════════════════════════════════════════════════════
# ⑥ VALIDACIÓN FINAL
# ══════════════════════════════════════════════════════════════════════════════

def _validar(df: pd.DataFrame, target: str, contaminantes: list[str]) -> None:
    """
    Verifica las tres invariantes que garantizan la calidad del dataset ML.
    Lanza AssertionError si alguna se viola.
    """
    print("  Verificando invariantes del dataset final…")

    assert df[target].isna().sum() == 0, \
        f"FALLO: '{target}' tiene NaN residuales."
    print(f"  ✓  Target '{target}': 0 NaN")

    cols  = [c for c in contaminantes if c in df.columns]
    valid = df[cols].notna().any(axis=1)
    assert valid.all(), \
        "FALLO: existen filas sin ningún contaminante con dato."
    print("  ✓  Coherencia química: cada fila tiene ≥1 contaminante válido")

    assert df.index.is_monotonic_increasing, \
        "FALLO: el Timestamp no está ordenado."
    print("  ✓  Timestamp ordenado cronológicamente")

    n_dup = df.index.duplicated().sum()
    if n_dup:
        print(f"  ⚠  {n_dup} timestamps duplicados (revisa la fuente)")
    else:
        print("  ✓  Sin timestamps duplicados")

    print("  ✅  Dataset validado\n")


# ══════════════════════════════════════════════════════════════════════════════
# ⑦ PERFIL ESTADÍSTICO Y EXPORTACIÓN
# ══════════════════════════════════════════════════════════════════════════════

def _perfil(df: pd.DataFrame, target: str) -> None:
    """Imprime un perfil compacto del dataset final (sin features adicionales)."""
    sep = "═" * 78
    print(f"\n{sep}")
    print(f"  PERFIL DEL DATASET ML")
    print(f"{'─'*78}")
    print(f"  Filas    : {len(df):,}")
    print(f"  Columnas : {df.shape[1]}  →  {list(df.columns)}")
    print(f"  Inicio   : {df.index.min()}")
    print(f"  Fin      : {df.index.max()}")
    print(f"  Duración : {(df.index.max() - df.index.min()).days:,} días")
    print(f"{'─'*78}")
    print(f"  {'Variable':<22}  {'N válidos':>10}  {'% NaN':>7}  "
          f"{'Media':>10}  {'Mín':>10}  {'Máx':>10}")
    print(f"{'─'*78}")
    for col in df.select_dtypes(include=[np.number]).columns:
        nv   = df[col].notna().sum()
        pnan = df[col].isna().mean() * 100
        mark = " ← TARGET" if col == target else ""
        print(f"  {col:<22}  {nv:>10,}  {pnan:>6.1f}%  "
              f"{df[col].mean():>10.3f}  {df[col].min():>10.3f}  "
              f"{df[col].max():>10.3f}{mark}")
    print(f"{sep}\n")


def exportar_csv(df: pd.DataFrame, parroquia: str, output_dir: Path) -> Path:
    """
    Exporta el dataset a CSV.
    Incluye un bloque de metadatos comentado al inicio del archivo para
    documentar la trazabilidad del procesamiento.

    Leer el CSV desde Python:
        df = pd.read_csv('dataset_ml_belisario.csv',
                         comment='#',
                         index_col='Timestamp',
                         parse_dates=True)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    nombre = f"dataset_ml_{parroquia.strip().lower().replace(' ', '_')}.csv"
    ruta   = output_dir / nombre

    meta = [
        "# ═══════════════════════════════════════════════════════════════════",
        "# DATASET ML — Red REMMAQ / Quito",
        f"# Parroquia    : {parroquia.upper()}",
        f"# Target       : PM25 (µg/m³)",
        f"# Filas        : {len(df):,}",
        f"# Columnas     : {df.shape[1]}  →  {list(df.columns)}",
        f"# Inicio       : {df.index.min()}",
        f"# Fin          : {df.index.max()}",
        f"# NaN target   : 0  (garantizado por R1)",
        f"# NaN secundarios: preservados — HistGradientBoosting los maneja",
        f"# Generado     : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
        "#",
        "# Uso:",
        "#   df = pd.read_csv('dataset_ml_*.csv', comment='#',",
        "#                    index_col='Timestamp', parse_dates=True)",
        "#   X  = df.drop(columns=['PM25'])",
        "#   y  = df['PM25']",
        "# ═══════════════════════════════════════════════════════════════════",
    ]

    with open(ruta, "w", encoding="utf-8") as f:
        for l in meta:
            f.write(l + "\n")
        df.to_csv(f, date_format="%Y-%m-%d %H:%M")

    kb = ruta.stat().st_size / 1024
    print(f"  💾  Exportado → {ruta}  ({kb:.1f} KB)\n")
    return ruta


# ══════════════════════════════════════════════════════════════════════════════
# ⑧ PIPELINE COMPLETO
# ══════════════════════════════════════════════════════════════════════════════

def construir_dataset_ml(
    data_dir:          str | Path,
    parroquia:         str       = PARROQUIA_OBJETIVO,
    target:            str       = TARGET,
    contaminantes:     list[str] = CONTAMINANTES,
    excluir_pandemia:  bool      = EXCLUIR_PANDEMIA,
    umbral_gap_horas:  int       = UMBRAL_GAP_HORAS,
    rangos:            dict      = RANGOS_VALIDOS,
    output_dir:        str | Path = OUTPUT_DIR,
) -> pd.DataFrame:
    """
    Pipeline completo de construcción del dataset ML.

    Parámetros
    ----------
    data_dir         : carpeta donde están los archivos REMMAQ
                       (PM10.xlsx, SO2.xlsx, CO.xlsx, etc.)
    parroquia        : nombre de la estación a extraer de cada archivo
    target           : columna objetivo (PM25)
    contaminantes    : lista de variables de calidad del aire
    excluir_pandemia : si True, elimina 01-01-2020 → 31-12-2021
    umbral_gap_horas : horas de silencio para descartar un bloque (R3)
    rangos           : rangos físicos aceptables por variable
    output_dir       : carpeta donde se exporta el CSV final

    Retorna
    -------
    pd.DataFrame con índice Timestamp, listo para HistGradientBoosting.
    """
    data_dir = Path(data_dir)
    out      = Path(output_dir)
    sep      = "═" * 65

    print(f"\n{sep}")
    print(f"  DATASET ML — REMMAQ   |   Parroquia: {parroquia.upper()}")
    print(f"  Target: {target}   |   Gap umbral: {umbral_gap_horas}h")
    print(f"{sep}")

    # ── PASO 1: Detectar archivos disponibles ─────────────────────────────────
    print(f"\n▶  PASO 1: Buscando archivos en '{data_dir}'…\n")
    archivos = buscar_archivos_en_directorio(data_dir)

    if not archivos:
        raise FileNotFoundError(
            f"No se encontró ningún archivo reconocido en '{data_dir}'. "
            "Verifica la ruta."
        )

    # Avisar si falta el target
    if target not in archivos:
        print(f"\n  ⚠  ATENCIÓN: el archivo del target ('{target}') no se "
              f"encontró en '{data_dir}'.")
        print(f"     Sin el target, todas las filas serán eliminadas por R1.")
        print(f"     Archivos encontrados: {list(archivos.keys())}\n")

    print(f"\n  Archivos encontrados ({len(archivos)}):")
    for var, ruta in archivos.items():
        print(f"    {'✓':<3} {var:<22} → {ruta.name}")

    # ── PASO 2: Carga y outer merge ───────────────────────────────────────────
    print(f"\n▶  PASO 2: Cargando y uniendo archivos (outer merge)…\n")
    df = cargar_dataset_parroquia(archivos, parroquia)
    print(f"\n  Dataset bruto (post-merge): {df.shape[0]:,} filas × "
          f"{df.shape[1]} columnas")
    print(f"  Período: {df.index.min()} → {df.index.max()}")

    auditor = _Auditor(len(df))

    # ── PASO 3: Limpiar errores de sensor ────────────────────────────────────
    print(f"\n▶  PASO 3: Limpiando errores de sensor…")
    df = limpiar_errores_sensor(df, rangos)

    # ── PASO 4: Excluir pandemia (opcional) ───────────────────────────────────
    if excluir_pandemia:
        print(f"\n▶  PASO 4: Excluyendo pandemia…")
        df = _filtrar_pandemia(df, PANDEMIA_INICIO, PANDEMIA_FIN, auditor)
    else:
        print(f"\n▶  PASO 4: Pandemia conservada (excluir_pandemia=False).")

    # ── PASO 5: Reglas de coherencia ML ──────────────────────────────────────
    print(f"\n▶  PASO 5: Aplicando reglas de coherencia ML…\n")
    df = _r1_sin_target(df, target, auditor)
    df = _r2_sin_referencia_quimica(df, contaminantes, auditor)
    df = _r3_gaps_largos(df, contaminantes, umbral_gap_horas, auditor)

    auditor.resumen()

    if df.empty:
        raise RuntimeError(
            "El dataset quedó vacío tras el filtrado.\n"
            f"  • ¿Existe '{target}' en la carpeta '{data_dir}'?\n"
            f"  • ¿Tiene datos la parroquia '{parroquia}' en ese archivo?\n"
            "  Revisa PARROQUIA_OBJETIVO y la disponibilidad del archivo target."
        )

    # ── PASO 6: Validación de invariantes ────────────────────────────────────
    print(f"▶  PASO 6: Validando dataset…\n")
    _validar(df, target, contaminantes)

    # ── PASO 7: Perfil estadístico ────────────────────────────────────────────
    print(f"▶  PASO 7: Perfil estadístico del dataset limpio:")
    _perfil(df, target)

    # ── PASO 8: Añadir features para ML (lags + variables cíclicas) ──────────
    print(f"▶  PASO 8: Generando features para ML (lags PM2.5 + hora/mes cíclicos)…")
    df = _generar_features_ml(df, target)
    print(f"  ✓  Nuevas columnas añadidas: PM25_lag_1h, PM25_lag_3h, PM25_lag_24h, "
          f"hora_sin, hora_cos, mes_sin, mes_cos")
    print(f"  Columnas totales finales: {df.shape[1]}")

    # ── PASO 9: Exportar ──────────────────────────────────────────────────────
    print(f"\n▶  PASO 9: Exportando dataset…")
    exportar_csv(df, parroquia, out)

    print(f"{sep}")
    print(f"  ✅  Dataset listo para HistGradientBoosting")
    print(f"  Filas finales  : {len(df):,}")
    print(f"  NaN en target  : 0 (garantizado)")
    print(f"  NaN secundarios: preservados (HistGB los maneja de forma nativa)")
    print(f"{sep}\n")

    # ── Ejemplo de uso con sklearn ────────────────────────────────────────────
    features = [c for c in df.columns if c != target]
    print(f"  Integración con HistGradientBoosting:")
    print(f"  ─────────────────────────────────────────────────────────────")
    print(f"  from sklearn.ensemble import HistGradientBoostingRegressor")
    print(f"  from sklearn.model_selection import train_test_split")
    print(f"")
    print(f"  X = df[{features}]")
    print(f"  y = df['{target}']")
    print(f"  X_train, X_test, y_train, y_test = train_test_split(")
    print(f"      X, y, test_size=0.2, shuffle=False  # shuffle=False → orden temporal")
    print(f"  )")
    print(f"  model = HistGradientBoostingRegressor()  # soporta NaN nativamente")
    print(f"  model.fit(X_train, y_train)")
    print(f"  ─────────────────────────────────────────────────────────────\n")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# ⑨ PUNTO DE ENTRADA (interactivo o directo)
# ══════════════════════════════════════════════════════════════════════════════

def main() -> pd.DataFrame:
    """
    Modo interactivo: pregunta el directorio de datos y ejecuta el pipeline.
    Para cambiar de parroquia edita PARROQUIA_OBJETIVO al inicio del script.
    """
    print("\n" + "═" * 65)
    print("  DATASET ML — Red REMMAQ (Quito)")
    print("  Modelo objetivo: HistGradientBoostingRegressor")
    print("═" * 65)
    print(f"\n  Parroquia objetivo : {PARROQUIA_OBJETIVO}")
    print(f"  Target             : {TARGET}")
    print(f"  Excluir pandemia   : {EXCLUIR_PANDEMIA} (2020-01-01 → 2021-12-31)")
    print(f"  Umbral gap         : {UMBRAL_GAP_HORAS} h")

    data_dir_str = input(
        "\n  Directorio con los archivos REMMAQ\n"
        "  (PM10.xlsx, SO2.xlsx, CO.xlsx…)\n"
        "  [Enter = directorio actual]: "
    ).strip()
    data_dir = Path(data_dir_str) if data_dir_str else Path(".")

    out_str = input(
        "\n  Carpeta de salida del CSV\n"
        "  [Enter = './dataset_ml']: "
    ).strip()
    output_dir = Path(out_str) if out_str else Path(OUTPUT_DIR)

    return construir_dataset_ml(
        data_dir         = data_dir,
        parroquia        = PARROQUIA_OBJETIVO,
        target           = TARGET,
        contaminantes    = CONTAMINANTES,
        excluir_pandemia = EXCLUIR_PANDEMIA,
        umbral_gap_horas = UMBRAL_GAP_HORAS,
        rangos           = RANGOS_VALIDOS,
        output_dir       = output_dir,
    )


# ══════════════════════════════════════════════════════════════════════════════
# USO PROGRAMÁTICO (Jupyter / batch)
# ══════════════════════════════════════════════════════════════════════════════
#
# from dataset_maestro_remmaq import construir_dataset_ml
#
# df = construir_dataset_ml(
#     data_dir   = "/ruta/a/mis/archivos",   # carpeta con PM10.xlsx, SO2.xlsx…
#     parroquia  = "BELISARIO",              # cambia aquí para otra estación
#     output_dir = "dataset_ml",
# )
#
# ── Para otra parroquia sin editar el script ──────────────────────────────────
# df_cotocollao = construir_dataset_ml(
#     data_dir  = "/ruta/archivos",
#     parroquia = "COTOCOLLAO",
# )
#
# df_belisario = construir_dataset_ml(
#     data_dir  = "/ruta/archivos",
#     parroquia = "BELISARIO",
# )
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    main()


═════════════════════════════════════════════════════════════════
  DATASET ML — Red REMMAQ (Quito)
  Modelo objetivo: HistGradientBoostingRegressor
═════════════════════════════════════════════════════════════════

  Parroquia objetivo : BELISARIO
  Target             : PM25
  Excluir pandemia   : True (2020-01-01 → 2021-12-31)
  Umbral gap         : 24 h



═════════════════════════════════════════════════════════════════
  DATASET ML — REMMAQ   |   Parroquia: BELISARIO
  Target: PM25   |   Gap umbral: 24h
═════════════════════════════════════════════════════════════════

▶  PASO 1: Buscando archivos en '.'…


  Archivos encontrados (11):
    ✓   PM25                   → PM2.5.xlsx
    ✓   PM10                   → PM10.xlsx
    ✓   O3                     → O3.xlsx
    ✓   CO                     → CO.xlsx
    ✓   NO2                    → NO2.xlsx
    ✓   SO2                    → SO2.xlsx
    ✓   Temperatura            → TMP.xlsx
    ✓   Humedad                → HUM.xlsx
    ✓   Viento_Velocidad       → VEL.xlsx
    ✓   Viento_Direccion       → DIR.xlsx
    ✓   Precipitacion          → LLU.xlsx

▶  PASO 2: Cargando y uniendo archivos (outer merge)…



Version dos del codigo de la creacion de data set con interfaz grafica usando gradio echo por claude

In [ ]:
"""
================================================================================
  REMMAQ — Dataset ML Builder  |  Interfaz Gradio
  Autor : Senior Data Engineer
  Uso   : python remmaq_gradio.py
================================================================================
  Instalación de dependencias:
      pip install gradio pandas numpy openpyxl xlrd
================================================================================
"""

import io
import os
import shutil
import tempfile
import warnings
import numpy as np
import pandas as pd
import gradio as gr
from pathlib import Path

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTES
# ══════════════════════════════════════════════════════════════════════════════

CONTAMINANTES: list[str] = ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]
TARGET: str = "PM25"
PANDEMIA_INICIO = "2020-01-01"
PANDEMIA_FIN    = "2021-12-31"

RANGOS_VALIDOS: dict = {
    "PM25":             (0, 400),
    "PM10":             (0, 999),
    "O3":               (0, 500),
    "CO":               (0, 50),
    "NO2":              (0, 500),
    "SO2":              (0, 500),
    "Temperatura":      (-10, 50),
    "Humedad":          (0, 100),
    "Viento_Velocidad": (0, 50),
    "Viento_Direccion": (0, 360),
    "Precipitacion":    (0, 200),
}

NOMBRES_ARCHIVOS: dict = {
    "PM25":             ["PM2.5.xlsx", "PM25.xlsx", "PM25.csv"],
    "PM10":             ["PM10.xlsx",  "PM10.csv"],
    "O3":               ["O3.xlsx",    "O3.csv"],
    "CO":               ["CO.xlsx",    "CO.csv"],
    "NO2":              ["NO2.xlsx",   "NO2.csv"],
    "SO2":              ["SO2.xlsx",   "SO2.csv"],
    "Temperatura":      ["TMP.xlsx",   "Temperatura.xlsx", "Temperatura.csv"],
    "Humedad":          ["HUM.xlsx",   "Humedad.xlsx",     "Humedad.csv"],
    "Viento_Velocidad": ["VEL.xlsx",   "Viento_Velocidad.xlsx"],
    "Viento_Direccion": ["DIR.xlsx",   "Viento_Direccion.xlsx"],
    "Precipitacion":    ["LLU.xlsx",   "Precipitacion.xlsx"],
}

ALL_EXPECTED_NAMES = [n for nlist in NOMBRES_ARCHIVOS.values() for n in nlist]


# ══════════════════════════════════════════════════════════════════════════════
# LÓGICA DE PROCESAMIENTO
# ══════════════════════════════════════════════════════════════════════════════

def _leer_archivo(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == ".csv":
        for sep in (",", ";", "\t"):
            try:
                df = pd.read_csv(path, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
        raise ValueError(f"No se pudo leer el CSV: {path}")
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Extensión no soportada: {ext}")


def _preparar_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={df.columns[0]: "Fecha"})
    mask_u = df["Fecha"].astype(str).str.contains(
        r"unidad|unit|ug|mg|%|m/s|°|grados", case=False, na=False, regex=True
    )
    df = df[~mask_u].reset_index(drop=True)
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")
    df = df.dropna(subset=["Fecha"]).set_index("Fecha").sort_index()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def buscar_archivos(data_dir: Path) -> dict[str, Path]:
    encontrados = {}
    for variable, candidatos in NOMBRES_ARCHIVOS.items():
        for nombre in candidatos:
            ruta = data_dir / nombre
            if ruta.exists():
                encontrados[variable] = ruta
                break
    return encontrados


def obtener_parroquias(archivos: dict[str, Path]) -> list[str]:
    for var in ["PM25", "PM10", "O3", "CO", "NO2", "SO2"]:
        if var in archivos:
            try:
                df = _leer_archivo(archivos[var])
                df = _preparar_df(df)
                cols = [c for c in df.columns if c.upper() not in ("FECHA", "DATE")]
                if cols:
                    return sorted([c.upper() for c in cols])
            except Exception:
                continue
    return []


def _extraer_columna(path: Path, variable: str, parroquia: str, log: list) -> pd.Series:
    if not path.exists():
        log.append(f"  ✗  [{variable:<22}] No encontrado: {path.name}")
        return pd.Series(dtype=float, name=variable)

    df = _leer_archivo(path)
    df = _preparar_df(df)

    pu = parroquia.strip().upper()
    col_match = None
    for col in df.columns:
        if col.upper() == pu:
            col_match = col
            break
    if col_match is None:
        candidatos = [c for c in df.columns if pu in c.upper()]
        if candidatos:
            col_match = candidatos[0]
            log.append(f"  ⚠  [{variable:<22}] parcial → '{col_match}'")
    if col_match is None:
        log.append(f"  ✗  [{variable:<22}] '{parroquia}' no encontrada")
        return pd.Series(dtype=float, name=variable)

    serie = pd.to_numeric(df[col_match], errors="coerce")
    serie.name = variable
    n_val = serie.notna().sum()
    rango = f"{serie.index.min().date()} → {serie.index.max().date()}"
    log.append(f"  ✓  [{variable:<22}] {n_val:>8,} valores  |  {rango}")
    return serie


def construir_dataset(
    archivos: dict[str, Path],
    parroquia: str,
    excluir_pandemia: bool,
    umbral_gap: int,
) -> tuple[pd.DataFrame, str]:
    log: list[str] = []
    SEP = "═" * 62

    log.append(SEP)
    log.append(f"  REMMAQ · Dataset ML — Parroquia: {parroquia.upper()}")
    log.append(f"  Target: {TARGET}  |  Gap: {umbral_gap}h  |  Pandemia excluida: {excluir_pandemia}")
    log.append(SEP)
    log.append("")

    # Cargar series
    log.append("▶ PASO 1 · Cargando archivos")
    series = []
    for var, ruta in archivos.items():
        s = _extraer_columna(ruta, var, parroquia, log)
        if not s.empty:
            series.append(s)

    if not series:
        raise RuntimeError(f"No se encontró '{parroquia}' en ningún archivo.")

    df = pd.concat(series, axis=1, join="outer").sort_index()
    df.index.name = "Timestamp"
    log.append(f"\n  Dataset bruto: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    log.append(f"  Período: {df.index.min()} → {df.index.max()}")

    n_total = len(df)
    etapas = []

    def reg(etapa, n_a, n_d):
        elim = n_a - n_d
        pct = elim / n_total * 100 if n_total else 0
        etapas.append((etapa, elim))
        ico = "✂" if elim else "✓"
        log.append(f"  {ico}  {etapa:<46}  −{elim:>7,} ({pct:.1f}%)")

    # Limpiar sensores
    log.append("\n▶ PASO 2 · Limpiando errores de sensor")
    df_c = df.copy()
    n_errores = 0
    for col in df_c.columns:
        n0 = df_c[col].notna().sum()
        df_c.loc[df_c[col].isin([-999, -9999, 9999]), col] = np.nan
        rng = RANGOS_VALIDOS.get(col)
        if rng:
            vmin, vmax = rng
            df_c.loc[(df_c[col] < vmin) | (df_c[col] > vmax), col] = np.nan
        n_errores += n0 - df_c[col].notna().sum()
    df = df_c
    log.append(f"  ✓  {n_errores:,} valores anómalos → NaN")

    # Filtros
    log.append("\n▶ PASO 3 · Filtros de coherencia ML")

    if excluir_pandemia:
        n = len(df)
        mask = (df.index >= PANDEMIA_INICIO) & (df.index <= PANDEMIA_FIN)
        df = df[~mask].copy()
        reg(f"Pandemia ({PANDEMIA_INICIO}→{PANDEMIA_FIN})", n, len(df))

    # R1
    if TARGET not in df.columns:
        raise RuntimeError(f"Columna '{TARGET}' no encontrada. ¿Subiste PM2.5.xlsx o PM25.xlsx?")
    n = len(df)
    df = df.dropna(subset=[TARGET]).copy()
    reg("R1 — Sin target (PM25=NaN)", n, len(df))

    # R2
    cols = [c for c in CONTAMINANTES if c in df.columns]
    if len(cols) >= 2:
        n = len(df)
        df = df[df[cols].notna().any(axis=1)].copy()
        reg("R2 — Sin ningún contaminante (falla total)", n, len(df))

    # R3
    if cols:
        n = len(df)
        silencio = df[cols].isna().all(axis=1)
        cambio = silencio != silencio.shift()
        id_blq = cambio.cumsum()
        df_sil = df[silencio].copy()
        blqs_malos: set = set()
        if not df_sil.empty:
            df_sil["_blq_"] = id_blq[silencio]
            durs = df_sil.groupby("_blq_").apply(
                lambda g: (g.index.max() - g.index.min()).total_seconds() / 3600
            )
            blqs_malos = set(durs[durs > umbral_gap].index.tolist())
            if blqs_malos:
                log.append(f"\n  Gaps detectados (>{umbral_gap}h):")
                for bid in sorted(blqs_malos):
                    g2 = df_sil[df_sil["_blq_"] == bid]
                    log.append(f"    • {g2.index.min()} → {g2.index.max()} ({durs[bid]:.1f}h)")
            mascara = silencio & id_blq.isin(blqs_malos)
            df = df[~mascara].copy()
        reg(f"R3 — Gaps >{umbral_gap}h", n, len(df))

    # Resumen filtrado
    n_final = len(df)
    ret = n_final / n_total * 100 if n_total else 0
    log.append(f"\n{'─'*62}")
    log.append(f"  {'Filas originales':<46}  {n_total:>10,}")
    for e, elim in etapas:
        log.append(f"  {e:<46}  −{elim:>9,}")
    log.append(f"{'─'*62}")
    log.append(f"  {'Filas finales':<46}  {n_final:>10,}  ({ret:.1f}%)")
    log.append(SEP)

    if df.empty:
        raise RuntimeError("El dataset quedó vacío tras el filtrado.")

    # Validación
    assert df[TARGET].isna().sum() == 0, "FALLO: PM25 tiene NaN residuales."

    # Features ML
    log.append("\n▶ PASO 4 · Generando features ML")
    df["PM25_lag_1h"]  = df[TARGET].shift(1)
    df["PM25_lag_3h"]  = df[TARGET].shift(3)
    df["PM25_lag_24h"] = df[TARGET].shift(24)
    hour  = df.index.hour
    month = df.index.month
    df["hora_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hora_cos"] = np.cos(2 * np.pi * hour / 24)
    df["mes_sin"]  = np.sin(2 * np.pi * month / 12)
    df["mes_cos"]  = np.cos(2 * np.pi * month / 12)
    log.append(f"  ✓  +7 columnas (lags PM25 + cíclicas hora/mes)")
    log.append(f"  ✓  Total columnas: {df.shape[1]}")
    log.append(f"\n  ✅  Dataset listo — {len(df):,} filas · {df.shape[1]} columnas")
    log.append(SEP + "\n")

    return df, "\n".join(log)


def df_a_csv(df: pd.DataFrame, parroquia: str) -> str:
    meta = [
        "# ═══════════════════════════════════════════════════════════════",
        "# DATASET ML — Red REMMAQ / Quito",
        f"# Parroquia    : {parroquia.upper()}",
        f"# Target       : PM25 (µg/m³)",
        f"# Filas        : {len(df):,}",
        f"# Columnas     : {df.shape[1]}  →  {list(df.columns)}",
        f"# Inicio       : {df.index.min()}",
        f"# Fin          : {df.index.max()}",
        f"# Generado     : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}",
        "#",
        "# Uso: pd.read_csv('...csv', comment='#', index_col='Timestamp', parse_dates=True)",
        "# ═══════════════════════════════════════════════════════════════",
    ]
    buf = io.StringIO()
    for line in meta:
        buf.write(line + "\n")
    df.to_csv(buf, date_format="%Y-%m-%d %H:%M")
    return buf.getvalue()


# ══════════════════════════════════════════════════════════════════════════════
# ESTADO GLOBAL (sesión)
# ══════════════════════════════════════════════════════════════════════════════

_estado = {
    "archivos": {},
    "parroquias": [],
    "temp_dir": None,
}


# ══════════════════════════════════════════════════════════════════════════════
# FUNCIONES DE CALLBACK
# ══════════════════════════════════════════════════════════════════════════════

def escanear_directorio(directorio: str):
    """Escanea el directorio y actualiza el estado."""
    data_dir = Path(directorio.strip())
    if not data_dir.exists():
        return (
            gr.update(value=f"❌ El directorio `{directorio}` no existe.", visible=True),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    archivos = buscar_archivos(data_dir)
    if not archivos:
        return (
            gr.update(
                value=f"⚠️ No se encontraron archivos REMMAQ en `{directorio}`.\n"
                       "Usa la pestaña de **Subida manual**.",
                visible=True,
            ),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    _estado["archivos"] = archivos
    parroquias = obtener_parroquias(archivos)
    _estado["parroquias"] = parroquias

    msg = (
        f"✅ {len(archivos)} archivo(s) encontrado(s): "
        + ", ".join(archivos.keys())
    )
    return (
        gr.update(value=msg, visible=True),
        gr.update(choices=parroquias, value=parroquias[0] if parroquias else None),
        gr.update(value=_tabla_archivos(archivos)),
    )


def procesar_subida(files):
    """Guarda archivos subidos en un temp dir y escanea."""
    if not files:
        return (
            gr.update(value="⚠️ No se subieron archivos.", visible=True),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    if _estado["temp_dir"] is None:
        _estado["temp_dir"] = tempfile.mkdtemp()
    tmp = Path(_estado["temp_dir"])

    for f in files:
        src = Path(f.name)
        dest = tmp / src.name
        shutil.copy2(src, dest)

    archivos = buscar_archivos(tmp)
    _estado["archivos"] = archivos

    if not archivos:
        no_reconocidos = [Path(f.name).name for f in files]
        return (
            gr.update(
                value=f"❌ Ningún archivo reconocido. Nombres esperados: "
                      f"PM2.5.xlsx, PM10.xlsx, SO2.xlsx, etc. "
                      f"Recibidos: {', '.join(no_reconocidos)}",
                visible=True,
            ),
            gr.update(choices=[], value=None),
            gr.update(value=_tabla_archivos({})),
        )

    parroquias = obtener_parroquias(archivos)
    _estado["parroquias"] = parroquias

    msg = f"✅ {len(archivos)} archivo(s) reconocido(s): " + ", ".join(archivos.keys())
    return (
        gr.update(value=msg, visible=True),
        gr.update(choices=parroquias, value=parroquias[0] if parroquias else None),
        gr.update(value=_tabla_archivos(archivos)),
    )


def _tabla_archivos(archivos: dict) -> pd.DataFrame:
    rows = []
    for var, nlist in NOMBRES_ARCHIVOS.items():
        encontrado = var in archivos
        rows.append({
            "Variable": var,
            "Estado": "✓ Cargado" if encontrado else "✗ No encontrado",
            "Archivo": archivos[var].name if encontrado else "—",
        })
    return pd.DataFrame(rows)


def ejecutar_pipeline(
    parroquia: str,
    excluir_pandemia: bool,
    umbral_gap: int,
    progress=gr.Progress(track_tqdm=True),
):
    """Ejecuta el pipeline y retorna log, métricas, preview y ruta del CSV."""
    archivos = _estado.get("archivos", {})
    if not archivos:
        return (
            "❌ No hay archivos cargados. Primero escanea un directorio o sube archivos.",
            None,
            None,
            None,
        )
    if not parroquia:
        return (
            "❌ Selecciona o escribe una parroquia primero.",
            None,
            None,
            None,
        )

    try:
        progress(0.1, desc="Cargando archivos…")
        df, log_text = construir_dataset(
            archivos=archivos,
            parroquia=parroquia.strip().upper(),
            excluir_pandemia=excluir_pandemia,
            umbral_gap=int(umbral_gap),
        )
        progress(0.9, desc="Exportando CSV…")

        # Guardar CSV temporal
        tmp_out = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".csv",
            prefix=f"dataset_ml_{parroquia.lower().replace(' ','_')}_",
        )
        contenido = df_a_csv(df, parroquia)
        tmp_out.write(contenido.encode("utf-8"))
        tmp_out.close()

        # Perfil para mostrar
        perfil_rows = []
        for col in df.select_dtypes(include=[np.number]).columns:
            perfil_rows.append({
                "Variable": col,
                "N válidos": int(df[col].notna().sum()),
                "% NaN": round(df[col].isna().mean() * 100, 1),
                "Media": round(df[col].mean(), 3),
                "Mín": round(df[col].min(), 3),
                "Máx": round(df[col].max(), 3),
            })

        progress(1.0, desc="¡Listo!")
        return (
            log_text,
            pd.DataFrame(perfil_rows),
            df.head(100),
            tmp_out.name,
        )

    except Exception as e:
        return (str(e), None, None, None)


# ══════════════════════════════════════════════════════════════════════════════
# TEMA Y CSS PERSONALIZADO
# ══════════════════════════════════════════════════════════════════════════════

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600;700&display=swap');

/* ── Base ── */
body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #f0f4f8 !important;
}
.dark body, .dark .gradio-container {
    background: #0f1923 !important;
}

/* ── Header ── */
.remmaq-title {
    background: linear-gradient(135deg, #1a2b3c 0%, #0f4c75 100%);
    color: white;
    padding: 2rem 2.5rem;
    border-radius: 16px;
    margin-bottom: 1.5rem;
    border-left: 5px solid #00d4aa;
    box-shadow: 0 8px 32px rgba(0,212,170,0.15);
}
.remmaq-title h1 {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 2rem !important;
    font-weight: 700 !important;
    margin: 0 0 0.3rem !important;
    color: #00d4aa !important;
}
.remmaq-title p {
    color: #a8c7e0 !important;
    margin: 0 !important;
    font-size: 0.9rem !important;
}

/* ── Tabs ── */
.tab-nav button {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 0.9rem !important;
    border-radius: 8px 8px 0 0 !important;
    padding: 0.6rem 1.2rem !important;
}
.tab-nav button.selected {
    background: #0f4c75 !important;
    color: #00d4aa !important;
    border-bottom: 3px solid #00d4aa !important;
}

/* ── Botón principal ── */
#btn-ejecutar {
    background: linear-gradient(135deg, #00d4aa, #0f4c75) !important;
    color: white !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 0.75rem 2rem !important;
    box-shadow: 0 4px 15px rgba(0,212,170,0.3) !important;
    transition: all 0.25s ease !important;
}
#btn-ejecutar:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(0,212,170,0.45) !important;
}

/* ── Botón secundario ── */
#btn-escanear, #btn-subida {
    background: #1a2b3c !important;
    color: #00d4aa !important;
    border: 1.5px solid #00d4aa !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    transition: all 0.2s !important;
}
#btn-escanear:hover, #btn-subida:hover {
    background: #00d4aa !important;
    color: #0f1923 !important;
}

/* ── Log console ── */
#log-output textarea {
    font-family: 'Space Mono', monospace !important;
    font-size: 0.78rem !important;
    background: #0d1b2a !important;
    color: #00d4aa !important;
    border: 1px solid #1e3a5f !important;
    border-radius: 10px !important;
    line-height: 1.7 !important;
}

/* ── Inputs ── */
input[type="text"], input[type="number"] {
    border-radius: 8px !important;
    border: 1.5px solid #cbd5e1 !important;
    font-family: 'DM Sans', sans-serif !important;
}

/* ── Status ── */
#status-box textarea {
    border-radius: 10px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 500 !important;
}

/* ── Dataframe ── */
.svelte-1gfkzm4, table {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.83rem !important;
}

/* ── Card info ── */
.info-card {
    background: linear-gradient(135deg, #e8f4fd, #f0fdf9);
    border: 1px solid #bde0fe;
    border-left: 4px solid #00d4aa;
    border-radius: 10px;
    padding: 0.9rem 1.2rem;
    font-size: 0.85rem;
    color: #1a2b3c;
    margin-bottom: 0.75rem;
}

/* ── Accordion ── */
details summary {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    color: #1a2b3c !important;
}
"""

THEME = gr.themes.Base(
    primary_hue="cyan",
    secondary_hue="blue",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("DM Sans"), "sans-serif"],
).set(
    body_background_fill="#f0f4f8",
    block_background_fill="#ffffff",
    block_border_color="#e2e8f0",
    block_radius="12px",
    block_shadow="0 2px 12px rgba(0,0,0,0.06)",
    button_primary_background_fill="#00d4aa",
    button_primary_text_color="#0f1923",
    button_primary_border_color="#00d4aa",
    input_background_fill="#f8fafc",
)


# ══════════════════════════════════════════════════════════════════════════════
# CONSTRUCCIÓN DE LA INTERFAZ
# ══════════════════════════════════════════════════════════════════════════════

def build_app() -> gr.Blocks:
    with gr.Blocks(
        theme=THEME,
        css=CSS,
        title="REMMAQ · Dataset ML Builder",
    ) as app:

        # ── Header ────────────────────────────────────────────────────────────
        gr.HTML("""
        <div class="remmaq-title">
          <h1>🌫️ REMMAQ · Dataset ML Builder</h1>
          <p>Red Metropolitana de Monitoreo Atmosférico de Quito
             &nbsp;·&nbsp; Genera datasets listos para
             <code>HistGradientBoostingRegressor</code></p>
        </div>
        """)

        # ══════════════════════════════════════════════════════════════════════
        # FILA PRINCIPAL
        # ══════════════════════════════════════════════════════════════════════
        with gr.Row():

            # ── COLUMNA IZQUIERDA: Carga + Config ─────────────────────────────
            with gr.Column(scale=4):

                with gr.Tabs():

                    # ── Tab 1: Directorio ────────────────────────────────────
                    with gr.TabItem("📁 Directorio local"):
                        gr.Markdown(
                            "Ingresa la ruta donde están los archivos REMMAQ "
                            "(`.xlsx` / `.csv`) y pulsa **Escanear**."
                        )
                        with gr.Row():
                            txt_dir = gr.Textbox(
                                label="Ruta del directorio",
                                placeholder="/ruta/a/tus/archivos/remmaq",
                                scale=4,
                            )
                            btn_scan = gr.Button("🔍 Escanear", elem_id="btn-escanear", scale=1)

                    # ── Tab 2: Subida manual ─────────────────────────────────
                    with gr.TabItem("⬆️ Subir archivos"):
                        gr.Markdown(
                            "Sube los archivos REMMAQ. El nombre del archivo "
                            "debe coincidir con los esperados: "
                            "`PM2.5.xlsx`, `PM10.xlsx`, `SO2.xlsx`, `CO.xlsx`, "
                            "`O3.xlsx`, `NO2.xlsx`, `TMP.xlsx`, `HUM.xlsx`, "
                            "`VEL.xlsx`, `DIR.xlsx`, `LLU.xlsx`"
                        )
                        file_upload = gr.File(
                            label="Archivos REMMAQ",
                            file_count="multiple",
                            file_types=[".xlsx", ".xls", ".csv"],
                        )
                        btn_upload = gr.Button(
                            "📂 Procesar archivos subidos",
                            elem_id="btn-subida",
                        )

                # Estado
                status_box = gr.Textbox(
                    label="Estado",
                    interactive=False,
                    elem_id="status-box",
                    max_lines=3,
                    visible=False,
                )

                # Tabla de archivos detectados
                with gr.Accordion("📋 Archivos detectados", open=False):
                    tabla_archivos = gr.Dataframe(
                        value=_tabla_archivos({}),
                        interactive=False,
                        wrap=True,
                    )

                gr.Markdown("---")

                # ── Configuración ────────────────────────────────────────────
                gr.Markdown("### ⚙️ Configuración")

                parroquia_dd = gr.Dropdown(
                    label="Parroquia / Estación",
                    choices=[],
                    allow_custom_value=True,
                    info="Se detecta automáticamente al cargar archivos, o escribe el nombre",
                )

                excluir_pandemia = gr.Checkbox(
                    label="Excluir período pandemia COVID-19 (2020–2021)",
                    value=True,
                    info="Elimina datos 2020-01-01 → 2021-12-31 para evitar distorsiones",
                )

                umbral_gap = gr.Slider(
                    label="Umbral de gap (horas)",
                    minimum=6, maximum=72, value=24, step=6,
                    info="Elimina bloques donde todos los sensores están en silencio > N horas",
                )

                btn_run = gr.Button(
                    "▶  Construir Dataset ML",
                    variant="primary",
                    elem_id="btn-ejecutar",
                )

            # ── COLUMNA DERECHA: Resultados ────────────────────────────────
            with gr.Column(scale=6):

                with gr.Tabs():

                    # ── Tab: Log ──────────────────────────────────────────────
                    with gr.TabItem("🖥️ Log del pipeline"):
                        log_out = gr.Textbox(
                            label="",
                            lines=24,
                            max_lines=40,
                            interactive=False,
                            elem_id="log-output",
                            placeholder="El log aparecerá aquí al ejecutar el pipeline…",
                        )

                    # ── Tab: Perfil ──────────────────────────────────────────
                    with gr.TabItem("📊 Perfil estadístico"):
                        perfil_out = gr.Dataframe(
                            label="Estadísticas por variable",
                            interactive=False,
                            wrap=True,
                        )

                    # ── Tab: Preview ─────────────────────────────────────────
                    with gr.TabItem("👁️ Vista previa"):
                        preview_out = gr.Dataframe(
                            label="Primeras 100 filas",
                            interactive=False,
                            wrap=False,
                        )

                    # ── Tab: Descarga ────────────────────────────────────────
                    with gr.TabItem("⬇️ Descargar"):
                        gr.Markdown(
                            "### Descargar Dataset\n"
                            "El archivo CSV incluye metadatos en las primeras líneas "
                            "(comentadas con `#`) y está listo para usar con scikit-learn:"
                        )
                        gr.Markdown("""
```python
import pandas as pd
df = pd.read_csv('dataset_ml_*.csv',
                 comment='#',
                 index_col='Timestamp',
                 parse_dates=True)
X = df.drop(columns=['PM25'])
y = df['PM25']
```
""")
                        csv_download = gr.File(
                            label="Dataset generado (CSV)",
                            interactive=False,
                        )

        # ── FOOTER ────────────────────────────────────────────────────────────
        gr.Markdown(
            "<br><center style='color:#94a3b8;font-size:0.78rem;'>"
            "REMMAQ · Red Metropolitana de Monitoreo Atmosférico de Quito · "
            "Dataset para HistGradientBoostingRegressor (scikit-learn)"
            "</center>"
        )

        # ══════════════════════════════════════════════════════════════════════
        # EVENTOS
        # ══════════════════════════════════════════════════════════════════════

        # Escanear directorio
        btn_scan.click(
            fn=escanear_directorio,
            inputs=[txt_dir],
            outputs=[status_box, parroquia_dd, tabla_archivos],
        ).then(lambda: gr.update(visible=True), outputs=[status_box])

        # Subida manual
        btn_upload.click(
            fn=procesar_subida,
            inputs=[file_upload],
            outputs=[status_box, parroquia_dd, tabla_archivos],
        ).then(lambda: gr.update(visible=True), outputs=[status_box])

        # Ejecutar pipeline
        btn_run.click(
            fn=ejecutar_pipeline,
            inputs=[parroquia_dd, excluir_pandemia, umbral_gap],
            outputs=[log_out, perfil_out, preview_out, csv_download],
        )

    return app


# ══════════════════════════════════════════════════════════════════════════════
# PUNTO DE ENTRADA
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    app = build_app()
    app.launch(
        server_name="0.0.0.0",   # accesible en red local
        server_port=7860,
        share=False,             # True para un enlace público temporal de Gradio
        show_error=True,
        inbrowser=True,          # abre el navegador automáticamente
    )